# Apply Kept Asset Labels

Merge the filled kept-asset labeling template back into the kept asset manifest to create a real labeled hairstyle asset bank.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
import json
import pandas as pd

from app.ml.celeba_hair_rich import (
    load_review_table,
    merge_labels_into_kept_assets,
    read_jsonl,
    write_jsonl,
)

REVIEWED_ROOT = BACKEND_ROOT / 'data' / 'processed' / 'celeba_hair_rich_assets' / 'reviewed'
KEPT_JSONL = REVIEWED_ROOT / 'kept_assets.jsonl'
LABELING_CSV = REVIEWED_ROOT / 'labeling' / 'kept_asset_labeling_template.csv'
LABELED_JSONL = REVIEWED_ROOT / 'kept_assets_labeled.jsonl'
LABELED_CSV = REVIEWED_ROOT / 'kept_assets_labeled.csv'
LABELED_SUMMARY_JSON = REVIEWED_ROOT / 'kept_assets_labeled_summary.json'

REVIEWED_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon/backend/data/processed/celeba_hair_rich_assets/reviewed')

In [3]:
kept_assets = read_jsonl(KEPT_JSONL)
label_frame = load_review_table(LABELING_CSV)

print('Kept assets:', len(kept_assets))
print('Label rows:', len(label_frame))
label_frame.head(5)

Kept assets: 120
Label rows: 120


,asset_id,image_path,mask_path,gender_label,confidence_bucket,quality_score,suggested_gender,suggested_color,suggested_curl,suggested_bang,...,source_positive_fields,label_gender,label_length,label_curl,label_bang,label_volume,label_side_hair,label_color,label_style_family,label_notes
0,celeba_hair_000002,D:\Projects\Personal Projects\Hairstyle Recomm...,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,1.00,female,brown,wavy,NaN,...,"Brown_Hair,Wavy_Hair",female,long,wavy,NaN,high,full,brown,long_layered,sample_reference
1,celeba_hair_000003,D:\Projects\Personal Projects\Hairstyle Recomm...,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,0.96,male,black,straight,NaN,...,"Male,Black_Hair,Straight_Hair",male,short,straight,NaN,NaN,NaN,black,pompadour,sample_reference
2,celeba_hair_000004,D:\Projects\Personal Projects\Hairstyle Recomm...,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,0.96,male,black,NaN,NaN,...,"Male,Black_Hair",male,short,NaN,NaN,NaN,NaN,black,side_part,sample_reference
3,celeba_hair_000005,D:\Projects\Personal Projects\Hairstyle Recomm...,D:\Projects\Personal Projects\Hairstyle Recomm...,female,high,1.00,female,black,NaN,NaN,...,Black_Hair,female,medium,NaN,side_bangs,NaN,NaN,black,NaN,sample_reference
4,celeba_hair_000006,D:\Projects\Personal Projects\Hairstyle Recomm...,D:\Projects\Personal Projects\Hairstyle Recomm...,male,medium,0.96,male,black,straight,NaN,...,"Male,Black_Hair,Straight_Hair",male,medium,straight,NaN,NaN,NaN,black,NaN,sample_reference


In [4]:
labeled_assets, missing_labels = merge_labels_into_kept_assets(kept_assets, label_frame)

print('Labeled assets:', len(labeled_assets))
print('Missing labels:', len(missing_labels))
missing_labels[:10]

Labeled assets: 120
Missing labels: 0


[]

In [5]:
write_jsonl(labeled_assets, LABELED_JSONL)

rows = []
for item in labeled_assets:
    normalized = item['labeling']['normalized_attributes']
    rows.append({
        'asset_id': item['asset_id'],
        'gender_label': item['gender_label'],
        'confidence_bucket': item.get('confidence_bucket'),
        'quality_score': item.get('quality_score'),
        'length': normalized.get('length'),
        'curl': normalized.get('curl'),
        'bang': normalized.get('bang'),
        'volume': normalized.get('volume'),
        'side_hair': normalized.get('side_hair'),
        'color': normalized.get('color'),
        'style_family': normalized.get('style_family'),
        'labeling_status': item['labeling']['status'],
        'labeling_notes': item['labeling'].get('notes', ''),
    })

labeled_frame = pd.DataFrame(rows)
labeled_frame.to_csv(LABELED_CSV, index=False, encoding='utf-8')

summary = {
    'total_labeled_assets': len(labeled_assets),
    'missing_labels': len(missing_labels),
    'filled_length': int(labeled_frame['length'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_curl': int(labeled_frame['curl'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_bang': int(labeled_frame['bang'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_volume': int(labeled_frame['volume'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_side_hair': int(labeled_frame['side_hair'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_color': int(labeled_frame['color'].fillna('').astype(str).str.strip().ne('').sum()),
    'filled_style_family': int(labeled_frame['style_family'].fillna('').astype(str).str.strip().ne('').sum()),
}
LABELED_SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', LABELED_JSONL)
print('Wrote:', LABELED_CSV)
print('Wrote:', LABELED_SUMMARY_JSON)

Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled.jsonl
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled.csv
Wrote: D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\data\processed\celeba_hair_rich_assets\reviewed\kept_assets_labeled_summary.json


In [6]:
labeled_frame.head(20)

,asset_id,gender_label,confidence_bucket,quality_score,length,curl,bang,volume,side_hair,color,style_family,labeling_status,labeling_notes
0,celeba_hair_000002,female,high,1.00,long,wavy,None,high,full,brown,long_layered,reviewed_labeled,sample_reference
1,celeba_hair_000003,male,medium,0.96,short,straight,None,None,None,black,pompadour,reviewed_labeled,sample_reference
2,celeba_hair_000004,male,medium,0.96,short,None,None,None,None,black,side_part,reviewed_labeled,sample_reference
3,celeba_hair_000005,female,high,1.00,medium,None,side_bangs,None,None,black,None,reviewed_labeled,sample_reference
4,celeba_hair_000006,male,medium,0.96,medium,straight,None,None,None,black,None,reviewed_labeled,sample_reference
5,celeba_hair_000009,male,medium,1.00,short,straight,None,None,None,None,pompadour,reviewed_labeled,sample_reference
6,celeba_hair_000011,female,high,1.00,long,wavy,None,high,full,blonde,long_layered,reviewed_labeled,sample_reference
7,celeba_hair_000012,female,high,1.00,long,wavy,None,None,full,blonde,long_layered,reviewed_labeled,sample_reference
8,celeba_hair_000013,male,medium,0.96,long,None,None,None,full,black,long_layered,reviewed_labeled,sample_reference
9,celeba_hair_000014,female,high,1.00,long,wavy,None,None,full,blonde,long_layered,reviewed_labeled,sample_reference


In [7]:
pd.Series(summary)

total_labeled_assets    120
missing_labels            0
filled_length           117
filled_curl              86
filled_bang              10
filled_volume            18
filled_side_hair         66
filled_color             98
filled_style_family     101
dtype: int64